# Advanced RAG: ask questions about your own documents

This notebook builds a question-answering system over a PDF, one step at a time.

The idea behind RAG (Retrieval-Augmented Generation): instead of hoping a language model
already knows the answer, we first **find** the relevant pieces of our document, then hand
them to the model and ask it to answer **using only those pieces**. So the answers stay
tied to the source, and the model can say "I don't know" when the document does not cover it.

Each step lives in its own small file (`loader.py`, `embedder.py`, and so on). Here we just
call those functions and look at what comes back. This is the same code `main.py` runs.

**You need** a `.env` file in this folder with three keys:
`QDRANT_ENDPOINT`, `QDRANT_API_KEY`, and `GROQ_API_KEY`.

## Setup

Load the API keys and quiet down the libraries so each step's output stays easy to read.

In [1]:
# Quiet the noisy library logs and progress bars so we only see our own output.
import logging
from tqdm import tqdm
from functools import partialmethod

logging.basicConfig(level=logging.WARNING)
for noisy in ('httpx', 'sentence_transformers', 'transformers', 'pypdf'):
    logging.getLogger(noisy).setLevel(logging.ERROR)
tqdm.__init__ = partialmethod(tqdm.__init__, disable=True)  # hide the "Batches" bars

# Load QDRANT_* and GROQ_API_KEY from the .env file into the environment.
from dotenv import load_dotenv
load_dotenv()

# Settings (model names, search sizes) all live in config.py.
import config
print('Document folder:', config.DOCS_DIR)
print('Embedding model:', config.EMBEDDING_MODEL_NAME)
print('LLM (Groq)     :', config.GROQ_MODEL_NAME)

Document folder: /home/sourov/Desktop/adv_rag/document
Embedding model: all-MiniLM-L6-v2
LLM (Groq)     : llama-3.3-70b-versatile


## Step 1: Load the document

First we read the PDF. The loader gives back one record per page, holding the page's text,
the filename, and the page number. Empty pages are skipped.

In [2]:
from loader import load_directory

# Read every supported file in the document folder into page records.
records = load_directory(config.DOCS_DIR, config.SUPPORTED_EXTENSIONS)

print(f'Loaded {len(records)} pages')
print('First page preview:', records[0]['text'][:150])

Loaded 125 pages
First page preview: A PDF Reference for        The Complete Node.js Dev Course                Version 3.0


## Step 2: Split the pages into chunks

A whole page is too big to search well, so we cut each page into smaller pieces called
chunks. The chunks overlap a little, so a sentence split across a boundary still shows up
whole in the next chunk. This is what we will actually search over.

In [3]:
from splitter import split_records

# Cut the pages into overlapping chunks (sizes come from config.py).
chunks = split_records(records, chunk_size=config.CHUNK_SIZE, chunk_overlap=config.CHUNK_OVERLAP)

print(f'{len(records)} pages turned into {len(chunks)} chunks')
print('One chunk looks like:', chunks[12]['text'][:150])

/home/sourov/Desktop/adv_rag/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


125 pages turned into 406 chunks
One chunk looks like: Lesson 13: Destructuring and Property Shorthand Challenge.............................................. 37 
Lesson 14: Bonus: HTTP Requests Without a 


## Step 3: Turn each chunk into a vector

A computer cannot compare meaning directly, so we convert each chunk into a list of numbers
called a vector (an "embedding"). Chunks with similar meaning get similar vectors. We use a
small model that runs locally and produces 384 numbers per chunk.

In [4]:
from embedder import load_embedding_model, embed_chunks, embed_query

# Load the embedding model once.
embed_model = load_embedding_model()

# Add a vector to every chunk.
embed_chunks(embed_model, chunks)
vector_size = embed_model.get_embedding_dimension()

print('Numbers per vector:', vector_size)
print('First vector starts with:', chunks[0]['embedding'][:5])

Numbers per vector: 384
First vector starts with: [-0.08888815 -0.04633616 -0.04083881  0.00178206  0.07450218]


## Step 4: Store the vectors in a database

We send the vectors to Qdrant, a database built for searching vectors. Each chunk is stored
together with its text and page number, so when a search finds a vector we immediately get
the matching text back too.

In [5]:
from vector_store import connect_qdrant, create_collection, upsert_chunks, dense_search

# Connect, make a fresh collection sized to our vectors, and upload the chunks.
qdrant = connect_qdrant()
create_collection(qdrant, vector_size)
count = upsert_chunks(qdrant, chunks)

print(f'Stored {count} chunks in Qdrant')

Stored 406 chunks in Qdrant


## Step 5: Search by meaning

Now we can search. We turn the question into a vector with the same model, then ask Qdrant
for the closest chunks. Notice the top results are about Express even though they do not use
the exact words of the question. That is the point of searching by meaning.

In [6]:
# A sample question we will reuse below.
query = 'What is Express and how do we set up a server?'

# Turn the question into a vector and fetch the 3 closest chunks.
dense_hits = dense_search(qdrant, embed_query(embed_model, query), top_k=3)

for rank, hit in enumerate(dense_hits, 1):
    print(f"[{rank}] score={hit['score']:.3f} | page {hit['page']}")
    print('   ', hit['text'][:110].replace(chr(10), ' '))

[1] score=0.684 | page 37
    create your first web server with Express. Once the server is up and running, users will be  able to interact 
[2] score=0.637 | page 37
    Links  • Node.js http documentation  • Node.js https documentation  Section 7: Web Servers  Lesson 1: Section 
[3] score=0.560 | page 38
    Version 1.0 39  Express 101  To get started, add Express to your project.  npm i express@4.16.4  Next, you can


## Step 6: Search by exact words (BM25)

Searching by meaning can miss exact text like a version number or a package name. BM25 is
an older method that scores chunks by exact word overlap. It is good at the things meaning
search misses, so we keep both. Notice it returns different chunks than step 5.

In [7]:
from keyword_index import build_bm25, bm25_search

# Build the keyword index once, then search it.
bm25 = build_bm25(chunks)
kw_hits = bm25_search(bm25, chunks, 'npm install express@4.16.4', top_k=3)

for rank, hit in enumerate(kw_hits, 1):
    print(f"[{rank}] score={hit['score']:.2f} | page {hit['page']}")
    print('   ', hit['text'][:110].replace(chr(10), ' '))

[1] score=8.76 | page 11
    {    "name": "notes-app",    "version": "1.0.0",    "description": "",    "main": "app.js",    "scripts": {   
[2] score=8.44 | page 57
    Version 1.0 58  You can create a dev script with the value nodemon src/app.js -e js,hbs. This will start  up t
[3] score=8.27 | page 12
    Version 1.0 13  npm install validator@10.8.0  The command above installs version 10.8.0 of validator. If you w


## Step 7: Combine the two searches

We now have two rankings (meaning and exact words) with scores on different scales, so we
cannot just add them. Instead we combine them by **position**: a chunk near the top of both
lists ends up on top of the combined list. This is called Reciprocal Rank Fusion.

One thing to watch: this rewards chunks that *both* searches liked, which is not always the
single best chunk. Step 8 cleans that up.

In [8]:
from hybrid import reciprocal_rank_fusion

# Get a wider pool from each search, then fuse the two rankings into one.
dense = dense_search(qdrant, embed_query(embed_model, query), top_k=config.RETRIEVE_POOL)
keyword = bm25_search(bm25, chunks, query, top_k=config.RETRIEVE_POOL)
fused = reciprocal_rank_fusion(dense, keyword, top_k=5)

for rank, hit in enumerate(fused, 1):
    print(f"[{rank}] page {hit['page']}: {hit['text'][:90].replace(chr(10), ' ')}")

[1] page 82: have more control over how your server processes requests. This will be used to check  tha
[2] page 41: Documentation Links  • path  Lesson 5: Serving up CSS, JS, Images, and More  In this lesso
[3] page 108: Version 1.0 109  Lesson 4: Getting Started with Socket.io  In this lesson, you’ll install 
[4] page 76: endpoints in a single file is a fine way to get started, but that won’t scale well as you 
[5] page 76: router.post('/someEndpoint', (req, res) => {      // Do something  })    module.exports = 


## Step 8: Rerank to get the order right

The combine step only looked at positions, never at the question itself. A reranker fixes
that: it reads the question together with each chunk and scores how well they actually match.
It is slower, so we only run it on the small pool from step 7. The best chunk should now move
to the top with a clear lead.

In [9]:
from reranker import load_reranker, rerank

# Score each chunk against the question and keep the best few.
reranker = load_reranker()
pool = reciprocal_rank_fusion(dense, keyword, top_k=config.RETRIEVE_POOL)
reranked = rerank(reranker, query, pool, top_k=config.FINAL_TOP_K)

for rank, hit in enumerate(reranked, 1):
    print(f"[{rank}] score={hit['rerank_score']:.2f} | page {hit['page']}")
    print('   ', hit['text'][:110].replace(chr(10), ' '))

[1] score=5.00 | page 37
    create your first web server with Express. Once the server is up and running, users will be  able to interact 
[2] score=4.87 | page 38
    Version 1.0 39  Express 101  To get started, add Express to your project.  npm i express@4.16.4  Next, you can
[3] score=3.63 | page 41
    Documentation Links  • path  Lesson 5: Serving up CSS, JS, Images, and More  In this lesson, you’ll use the Ex
[4] score=3.47 | page 37
    Links  • Node.js http documentation  • Node.js https documentation  Section 7: Web Servers  Lesson 1: Section 
[5] score=2.43 | page 82
    have more control over how your server processes requests. This will be used to check  that a user is authenti


## Step 9: Write the answer

Finally we give the best chunks to the language model and ask it to answer. The rules baked
into the prompt: use only the provided chunks, and cite the page. The sources are printed so
you can check the answer against the document yourself.

In [10]:
from generator import connect_groq, generate_answer

# Send the top chunks to the LLM and get a grounded, cited answer.
groq = connect_groq()
answer = generate_answer(groq, query, reranked)

print(answer)
print()
print('Based on pages:', [h['page'] for h in reranked])

Express is a framework used to create a web server, allowing users to interact with an application via a browser [page 37]. To set up a server, you first need to add Express to your project by running the command `npm i express@4.16.4` [page 38]. Then, you can require Express and create a new application by calling the `express()` function: `const express = require('express')` and `const app = express()` [page 38]. This `app` object can be used to set up the server, including defining routes such as the home page and weather page using `app.get()` [page 38].

Based on pages: [37, 38, 41, 37, 82]


## All steps in one call

`main.py` bundles the steps into two functions: `build_index()` does steps 1 to 4, and
`answer_question()` does steps 5 to 9. We already built everything above, so we can ask a
new question in a single call.

In [11]:
from main import answer_question

# Same pipeline, one line. Reuses the pieces we built above.
answer, sources = answer_question(
    'How do I set up an Express server with npm?',
    embed_model, qdrant, bm25, chunks, reranker, groq,
)
print(answer)

To set up an Express server with npm, you need to add Express to your project by running the command `npm i express@4.16.4` [page 38]. Then, you can require Express and create a new Express application by calling the function `const app = express()` [page 38]. This will allow you to set up your server and define routes, such as the home page and weather page, using `app.get()` [page 38].


## Why this is trustworthy

Last test: ask something the document does not cover. A normal chatbot would make up an
answer. Ours refuses, because the answer is not in the chunks we retrieved. That refusal is
the whole reason RAG is useful for real documents.

In [12]:
# Ask an off-topic question. Expected reply: it does not know.
answer, sources = answer_question(
    'Who is the president of France?',
    embed_model, qdrant, bm25, chunks, reranker, groq,
)
print(answer)

I don't know based on the document.
